# Data Validation and Cleanup
## Project: Health Colombia - ETL

This notebook performs validation and cleanup of datasets before the ETL process.

**Datasets:**
- Healthcare system affiliates by department, municipality, and regime
- Public and private healthcare facilities by care level and installed capacity

In [ ]:
import pandas as pd
import numpy as np
import os
import sys

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

print(f"Python: {sys.version}")
print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")

## 1. Data Loading

In [ ]:
DATA_DIR = '../data/raw'

AFFILIATES_FILE = 'affiliates_by_department_municipality_regime_20260906.csv'
FACILITIES_FILE = 'healthcare_facilities_by_level_capacity_20260906.csv'

print(f"Loading datasets from: {DATA_DIR}")
print(f"Affiliates: {AFFILIATES_FILE}")
print(f"Facilities: {FACILITIES_FILE}")

In [ ]:
df_affiliates = pd.read_csv(f"{DATA_DIR}/{AFFILIATES_FILE}", dtype=str)
df_facilities = pd.read_csv(f"{DATA_DIR}/{FACILITIES_FILE}", dtype=str)

print("Datasets loaded successfully")
print(f"Affiliates: {df_affiliates.shape[0]:,} rows, {df_affiliates.shape[1]} columns")
print(f"Facilities: {df_facilities.shape[0]:,} rows, {df_facilities.shape[1]} columns")

## 2. Exploratory Analysis - Affiliates Dataset

In [ ]:
print("=" * 60)
print("DATASET: HEALTHCARE SYSTEM AFFILIATES")
print("=" * 60)

print("\nFirst 5 rows:")
df_affiliates.head()

In [ ]:
print("\nGeneral dataset info:")
df_affiliates.info()

In [ ]:
print("\nDescriptive statistics:")
df_affiliates.describe(include='all')

In [ ]:
print("\nData types per column:")
for col in df_affiliates.columns:
    print(f"  {col}: {df_affiliates[col].dtype} - Example: '{df_affiliates[col].iloc[0]}'")

### 2.1 Null Values - Affiliates

In [ ]:
print("\nNull values per column:")
null_affiliates = df_affiliates.isnull().sum()
null_percentage = (null_affiliates / len(df_affiliates) * 100).round(2)

df_null_aff = pd.DataFrame({
    'Nulls': null_affiliates,
    'Percentage': null_percentage
})
df_null_aff[df_null_aff['Nulls'] > 0].sort_values('Percentage', ascending=False)

In [ ]:
print("\nTotal null values:", df_affiliates.isnull().sum().sum())
print("Rows with at least one null:", df_affiliates.isnull().any(axis=1).sum())

### 2.2 Duplicates - Affiliates

In [ ]:
print("\nDuplicates per column:")
for col in df_affiliates.columns:
    dup_count = df_affiliates[col].duplicated().sum()
    if dup_count > 0:
        print(f"  {col}: {dup_count:,} duplicates")

In [ ]:
complete_duplicates = df_affiliates.duplicated().sum()
print(f"\nFully duplicated rows: {complete_duplicates:,}")
print(f"Percentage: {complete_duplicates/len(df_affiliates)*100:.2f}%")

if complete_duplicates > 0:
    print("\nExample of duplicated rows:")
    display(df_affiliates[df_affiliates.duplicated(keep=False)].head(10))

### 2.3 Unique Values - Affiliates

In [ ]:
print("\nUnique values per column:")
for col in df_affiliates.columns:
    unique_count = df_affiliates[col].nunique()
    print(f"  {col}: {unique_count:,} unique values")

In [ ]:
print("\nRegime distribution:")
print(df_affiliates['IDRegimen'].value_counts())

In [ ]:
print("\nDepartments with most affiliates:")
dept_count = df_affiliates['Departamento'].value_counts().head(10)
print(dept_count)

### 2.4 NumPersonas Analysis - Affiliates

In [ ]:
print("\nNumPersonas analysis:")
print(f"  Current type: {df_affiliates['NumPersonas'].dtype}")
print(f"  Example values: {df_affiliates['NumPersonas'].head(10).tolist()}")

df_affiliates['NumPersonas_Clean'] = df_affiliates['NumPersonas'].str.replace('.', '', regex=False)
df_affiliates['NumPersonas_Clean'] = pd.to_numeric(df_affiliates['NumPersonas_Clean'], errors='coerce')

print(f"\nAfter cleanup:")
print(f"  Null values: {df_affiliates['NumPersonas_Clean'].isnull().sum()}")
print(f"  Minimum: {df_affiliates['NumPersonas_Clean'].min():,.0f}")
print(f"  Maximum: {df_affiliates['NumPersonas_Clean'].max():,.0f}")
print(f"  Average: {df_affiliates['NumPersonas_Clean'].mean():,.0f}")

## 3. Exploratory Analysis - Facilities Dataset

In [ ]:
print("=" * 60)
print("DATASET: PUBLIC AND PRIVATE HEALTHCARE FACILITIES")
print("=" * 60)

print("\nFirst 5 rows:")
df_facilities.head()

In [ ]:
print("\nGeneral dataset info:")
df_facilities.info()

In [ ]:
print("\nDescriptive statistics:")
df_facilities.describe(include='all')

In [ ]:
print("\nData types per column:")
for col in df_facilities.columns:
    print(f"  {col}: {df_facilities[col].dtype} - Example: '{df_facilities[col].iloc[0]}'")

### 3.1 Null Values - Facilities

In [ ]:
print("\nNull values per column:")
null_facilities = df_facilities.isnull().sum()
null_percentage_fac = (null_facilities / len(df_facilities) * 100).round(2)

df_null_fac = pd.DataFrame({
    'Nulls': null_facilities,
    'Percentage': null_percentage_fac
})
df_null_fac[df_null_fac['Nulls'] > 0].sort_values('Percentage', ascending=False)

In [ ]:
print("\nTotal null values:", df_facilities.isnull().sum().sum())
print("Rows with at least one null:", df_facilities.isnull().any(axis=1).sum())

### 3.2 Duplicates - Facilities

In [ ]:
print("\nDuplicates per column:")
for col in df_facilities.columns:
    dup_count = df_facilities[col].duplicated().sum()
    if dup_count > 0:
        print(f"  {col}: {dup_count:,} duplicates")

In [ ]:
complete_duplicates_fac = df_facilities.duplicated().sum()
print(f"\nFully duplicated rows: {complete_duplicates_fac:,}")
print(f"Percentage: {complete_duplicates_fac/len(df_facilities)*100:.2f}%")

if complete_duplicates_fac > 0:
    print("\nExample of duplicated rows:")
    display(df_facilities[df_facilities.duplicated(keep=False)].head(10))

### 3.3 Unique Values - Facilities

In [ ]:
print("\nUnique values per column:")
for col in df_facilities.columns:
    unique_count = df_facilities[col].nunique()
    print(f"  {col}: {unique_count:,} unique values")

In [ ]:
print("\nNature distribution:")
print(df_facilities['naturaleza'].value_counts())

In [ ]:
print("\nCare level distribution:")
print(df_facilities['num nivel atencion'].value_counts())

### 3.4 Installed Capacity Analysis - Facilities

In [ ]:
print("\nInstalled capacity analysis:")
df_facilities['capacity_clean'] = pd.to_numeric(df_facilities['num cantidad capacidad instalada'], errors='coerce')

print(f"  Null values: {df_facilities['capacity_clean'].isnull().sum()}")
print(f"  Minimum: {df_facilities['capacity_clean'].min():,.0f}")
print(f"  Maximum: {df_facilities['capacity_clean'].max():,.0f}")
print(f"  Average: {df_facilities['capacity_clean'].mean():,.2f}")

In [ ]:
print("\nCapacity group distribution:")
print(df_facilities['nom grupo capacidad '].value_counts())

## 4. Data Quality Summary

In [ ]:
print("=" * 60)
print("DATA QUALITY SUMMARY")
print("=" * 60)

print("\nAFFILIATES:")
print(f"  Total records: {len(df_affiliates):,}")
print(f"  Complete duplicates: {df_affiliates.duplicated().sum():,}")
print(f"  Total null values: {df_affiliates.isnull().sum().sum():,}")
print(f"  Columns: {list(df_affiliates.columns)}")

print("\nFACILITIES:")
print(f"  Total records: {len(df_facilities):,}")
print(f"  Complete duplicates: {df_facilities.duplicated().sum():,}")
print(f"  Total null values: {df_facilities.isnull().sum().sum():,}")
print(f"  Columns: {list(df_facilities.columns)}")

## 5. Data Cleanup

In [ ]:
print("=" * 60)
print("CLEANUP PROCESS")
print("=" * 60)

df_affiliates_clean = df_affiliates.copy()
df_facilities_clean = df_facilities.copy()

### 5.1 Cleanup - Affiliates

In [ ]:
print("\nAffiliates dataset cleanup:")
print(f"  Initial records: {len(df_affiliates_clean):,}")

# Remove temporary column
df_affiliates_clean = df_affiliates_clean.drop(columns=['NumPersonas_Clean'])

# Clean NumPersonas
df_affiliates_clean['NumPersonas'] = df_affiliates_clean['NumPersonas'].str.replace('.', '', regex=False)
df_affiliates_clean['NumPersonas'] = pd.to_numeric(df_affiliates_clean['NumPersonas'], errors='coerce').fillna(0).astype(int)

# Convert numeric types
df_affiliates_clean['Año'] = df_affiliates_clean['Año'].astype(int)
df_affiliates_clean['Mes'] = df_affiliates_clean['Mes'].astype(int)

# Remove records with 0 affiliates
df_affiliates_clean = df_affiliates_clean[df_affiliates_clean['NumPersonas'] > 0]

# Remove duplicates
df_affiliates_clean = df_affiliates_clean.drop_duplicates()

print(f"  Final records: {len(df_affiliates_clean):,}")
print(f"  Records removed: {len(df_affiliates) - len(df_affiliates_clean):,}")

In [ ]:
print("\nPost-cleanup verification (Affiliates):")
print(f"  Duplicates: {df_affiliates_clean.duplicated().sum()}")
print(f"  Nulls: {df_affiliates_clean.isnull().sum().sum()}")
print(f"  Negative num_persons: {(df_affiliates_clean['NumPersonas'] < 0).sum()}")

### 5.2 Cleanup - Facilities

In [ ]:
print("\nFacilities dataset cleanup:")
print(f"  Initial records: {len(df_facilities_clean):,}")

# Remove temporary column
df_facilities_clean = df_facilities_clean.drop(columns=['capacity_clean'])

# Clean NIT (remove commas)
df_facilities_clean['nit IPS '] = df_facilities_clean['nit IPS '].str.replace(',', '', regex=False)

# Convert care level to numeric
df_facilities_clean['num nivel atencion'] = pd.to_numeric(df_facilities_clean['num nivel atencion'], errors='coerce')

# Clean installed capacity
df_facilities_clean['num cantidad capacidad instalada'] = pd.to_numeric(
    df_facilities_clean['num cantidad capacidad instalada'], errors='coerce'
).fillna(0).astype(int)

# Remove records without provider code or name
df_facilities_clean = df_facilities_clean.dropna(subset=['Código prestador', 'Nombre prestador'])

# Remove duplicates
df_facilities_clean = df_facilities_clean.drop_duplicates()

print(f"  Final records: {len(df_facilities_clean):,}")
print(f"  Records removed: {len(df_facilities) - len(df_facilities_clean):,}")

In [ ]:
print("\nPost-cleanup verification (Facilities):")
print(f"  Duplicates: {df_facilities_clean.duplicated().sum()}")
print(f"  Total nulls: {df_facilities_clean.isnull().sum().sum()}")
print(f"  Without provider code: {df_facilities_clean['Código prestador'].isnull().sum()}")

## 6. Save Clean Data

In [ ]:
OUTPUT_DIR = '../data/processed'
os.makedirs(OUTPUT_DIR, exist_ok=True)

df_affiliates_clean.to_csv(f"{OUTPUT_DIR}/affiliates_clean.csv", index=False, encoding='utf-8')
df_facilities_clean.to_csv(f"{OUTPUT_DIR}/facilities_clean.csv", index=False, encoding='utf-8')

print(f"Data saved to: {OUTPUT_DIR}")
print(f"  affiliates_clean.csv: {len(df_affiliates_clean):,} records")
print(f"  facilities_clean.csv: {len(df_facilities_clean):,} records")

## 7. Final Validation

In [ ]:
print("=" * 60)
print("FINAL VALIDATION")
print("=" * 60)

print("\nAffiliates Dataset (clean):")
print(df_affiliates_clean.info())

print("\n" + "=" * 60)
print("\nFacilities Dataset (clean):")
print(df_facilities_clean.info())

In [ ]:
print("\nClean data sample - Affiliates:")
df_affiliates_clean.head(10)

In [ ]:
print("\nClean data sample - Facilities:")
df_facilities_clean.head(10)

## Summary

### Validation and Cleanup Results

| Dataset | Original Records | Final Records | Removed |
|---------|------------------|---------------|----------|
| Affiliates | {:,} | {:,} | {:,} |
| Facilities | {:,} | {:,} | {:,} |

### Actions Performed
1. Data type conversion
2. Numeric field cleanup (periods as thousand separators)
3. Duplicate removal
4. Removal of records with nulls in critical fields
5. Removal of records with invalid values (0 affiliates)
6. Clean data saved to `data/processed/`